In [2]:
"""Train the LOF detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.lof import LOF
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
lof = LOF(
    contamination=config.CONTAMINATION,
    n_neighbors=config.N_NEIGHBORS,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
lof.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = lof.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    lof.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="lof",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


Epoch 1/50


2026-08-14 19:17:49.665010: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 4s 810us/step - loss: 8.5955e-04 - val_loss: 0.0011
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 779us/step - loss: 8.4079e-04 - val_loss: 0.0011
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 794us/step - loss: 8.3761e-04 - val_loss: 0.0011
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 800us/step - loss: 8.3754e-04 - val_loss: 0.0011
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 822us/step - loss: 8.3731e-04 - val_loss: 0.0011
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 793us/step - loss: 8.3601e-04 - val_loss: 0.0011
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 777us/step - loss: 8.3597e-04 - val_loss: 0.0011
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 779us/step - loss: 8.3533e-04 - val_loss: 0.0011
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 786us/step - loss: 8.3502e-04 - val_loss: 0.0011
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 771us/step - loss: 8.3476e-04 - val_loss: 0.0011
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 784us/step - loss: 8.378

INFO:tensorflow:Assets written to: /tmp/tmpo8drtwhs/assets


Saved artifact at '/tmp/tmpo8drtwhs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140633974193808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633964138896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633964137552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633964136592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633964139472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140633964136208: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786728016.767810   43450 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786728016.767819   43450 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 19:20:16.767990: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpo8drtwhs
2026-08-14 19:20:16.768293: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 19:20:16.768296: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpo8drtwhs
I0000 00:00:1786728016.770384   43450 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 19:20:16.770660: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>